In [1]:
"""
Load Lowe's sustainability CSV into MySQL.
Usage: python scripts/load_sustainability.py
"""

import csv
import mysql.connector

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_PASSWORD",
    "database": "lowes_db",
}
CSV_PATH = "data/raw/lowes_sustainability.csv"

def clean(value, cast_fn=str):
    if value is None:
        return None
    value = value.strip()
    if value == "":
        return None
    return cast_fn(value)

INSERT_SQL = """
    INSERT IGNORE INTO sustainability
        (id, report_year, scope1_emissions, scope2_emissions, scope2_market,
         scope3_emissions, total_energy_mwh, renewable_energy_pct, 
         water_consumption_bgal, waste_diverted_pct, recycling_rate_pct, 
         solar_energy_mwh, smartway_award, source_report)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    count = 0
    for row in reader:
        values = (
            clean(row["id"], int),
            clean(row["report_year"], int),
            clean(row["scope1_emissions"], float),
            clean(row["scope2_emissions"], float),
            clean(row["scope2_market"], float),
            clean(row["scope3_emissions"], float), 
            clean(row["total_energy_mwh"], float),
            clean(row["renewable_energy_pct"], float),
            clean(row["water_consumption_bgal"], float),
            clean(row["waste_diverted_pct"], float),      # ← ADD THIS (9th position)
            clean(row["recycling_rate_pct"], float),      # ← ADD THIS (10th position) 
            clean(row["solar_energy_mwh"], int),
            clean(row["smartway_award"], int),
            clean(row["source_report"]),
        )
        cursor.execute(INSERT_SQL, values)
        count += 1

conn.commit()
print(f"✓ Loaded {count} rows into stg_sustainability.")
cursor.close()
conn.close()

✓ Loaded 3 rows into stg_sustainability.
